# Article 1 — SAM 3.1 recorded-output runner

Run the perception stage on a Colab GPU and download a portable artifact bundle for the static evidence explorer. The notebook never embeds tokens, model weights, or the source video in Git.

Before running: select a **GPU** runtime, request access to `facebook/sam3`, and add `HF_TOKEN` plus a read-only `GITHUB_TOKEN` under Colab **Secrets**.

In [ ]:
import platform
import subprocess
import sys

print(f"Python: {platform.python_version()}")
assert sys.version_info >= (3, 12), "SAM 3.1 requires Python 3.12+"
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from base64 import b64encode
from pathlib import Path

from google.colab import userdata

ROOT = Path("/content")
GVI = ROOT / "grounded-visual-intelligence"
SAM3 = ROOT / "sam3"

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Add a read-only GITHUB_TOKEN in Colab Secrets")
basic = b64encode(f"x-access-token:{github_token}".encode()).decode()

if not GVI.exists():
    subprocess.run(
        [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {basic}",
            "clone",
            "--branch",
            "article/01-grounded-video-memory",
            "--depth",
            "1",
            "https://github.com/vlada22/grounded-visual-intelligence.git",
            str(GVI),
        ],
        check=True,
    )
if not SAM3.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/facebookresearch/sam3.git",
            str(SAM3),
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{GVI}[inference]",
        "-e",
        str(SAM3),
    ],
    check=True,
)

In [ ]:
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets after SAM 3 access is approved")
login(token=hf_token, add_to_git_credential=False)
print("Hugging Face authentication ready.")

In [ ]:
from google.colab import files

print("Upload one redistribution-safe MP4 (start with 5–15 seconds).")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one video")
video_path = Path("/content") / next(iter(uploaded))
print(f"Video: {video_path.name}")

In [ ]:
import json
from fractions import Fraction

probe = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height,avg_frame_rate,nb_frames,duration",
        "-of",
        "json",
        str(video_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
stream = json.loads(probe.stdout)["streams"][0]
fps = float(Fraction(stream["avg_frame_rate"]))
duration = float(stream.get("duration") or 0)
frame_count = int(stream.get("nb_frames") or round(duration * fps))
assert fps > 0 and frame_count > 0
video_info = {
    "width": int(stream["width"]),
    "height": int(stream["height"]),
    "fps": fps,
    "frame_count": frame_count,
}
video_info

In [ ]:
import time

import torch

from gvi.inference.sam3_worker import (
    Sam3InferenceWorker,
    VideoDescriptor,
    build_official_predictor,
)

PROMPT = "red toy car"  # Change this for the prepared scene.
SCENE_ID = "article-01-hero"
OUTPUT_DIR = ROOT / "gvi-artifacts" / SCENE_ID

descriptor = VideoDescriptor(
    scene_id=SCENE_ID,
    source_uri=f"prepared/{video_path.name}",
    **video_info,
)
predictor = build_official_predictor()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
recorded = Sam3InferenceWorker(predictor).run(
    video_path=video_path,
    video=descriptor,
    prompt=PROMPT,
    prompt_frame_index=0,
    output_dir=OUTPUT_DIR,
)
elapsed_s = time.perf_counter() - started
metrics = {
    "elapsed_s": elapsed_s,
    "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "frames": len(recorded.frames),
    "source_frames": descriptor.frame_count,
}
(OUTPUT_DIR / "run-metrics.json").write_text(json.dumps(metrics, indent=2))
metrics

In [ ]:
import shutil

archive = shutil.make_archive(str(ROOT / SCENE_ID), "zip", OUTPUT_DIR)
print(f"Artifact bundle: {archive}")
files.download(archive)

## Expected output

The ZIP contains `sam3-output.json`, `run-metrics.json`, and one portable RLE mask per object observation. Keep the video, tokens, and gated weights outside Git. We will ingest the recorded output into the Visual Evidence Graph and bind it to the web demo.